In [6]:
import pandas as pd
import torch
import torch.nn as nn
import re

import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk

try:
    nltk.download('punkt', quiet=True)
    nltk.download('stopwords', quiet=True)
except Exception as e:
    print(f"NLTK download failed: {e}")

In [7]:
df = pd.read_csv(r"C:\Users\pc\OneDrive\Music\Desktop\ML_Projects\Dataset\emails.csv")
df.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5728 entries, 0 to 5727
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    5728 non-null   object
 1   spam    5728 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 89.6+ KB


In [9]:
df.drop_duplicates(inplace=True)
df.shape

(5695, 2)

In [10]:
# Fuction for Removing Punchuation, html and http, 

def clean_data(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"\r\n", " ", text)
    text = re.sub(r"[^A-Za-z0-9\s]", " ", text) # Remove Punchuations 
    text = re.sub(r"http\S+", " ", text)    # Remove URLS
    text = re.sub(r"[<.*?>]", " ", text)    # Remove HTTP 
    text = text.strip().lower()             # Convert into Lower
    return text

df["text"] = df["text"].apply(clean_data)

In [11]:
# Remove Stopwords

def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")
    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")
    return text
df["text"] = df["text"].apply(remove_stopwords)

In [12]:
# Stemming

def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []
    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)
    return " ".join(stemmed_words)

df["text"] = df["text"].apply(stemming)

In [13]:
# Encoding & Vectorization

le = LabelEncoder()
y = df["spam"] = le.fit_transform(df["spam"])
df["spam"].value_counts()

spam
0    4327
1    1368
Name: count, dtype: int64

In [14]:
# TF-IDF Vectorization

tf = TfidfVectorizer(max_features=5000)
x = tf.fit_transform(df["text"])

In [15]:
# Train Test Split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)
x_train.shape, x_test.shape

((4556, 5000), (1139, 5000))

In [16]:
# Convert TensorDataset
x_train = x_train.toarray()
x_test  = x_test.toarray()

In [17]:
# Convert

train_set = TensorDataset(
    torch.from_numpy(x_train).float(),
    torch.from_numpy(y_train).float()
)

test_set = TensorDataset(
    torch.from_numpy(x_test).float(),
    torch.from_numpy(y_test).float()
)

In [18]:
#  Dataloader 

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_set, batch_size=64, shuffle=True)

In [19]:
import torch.nn as nn

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [20]:
input_size = x_train.shape[1]

model = RNN(input_size)

criteria = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [21]:
# Training Model

epochs = 20

for epoch in range (epochs):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        
        xb = xb.unsqueeze(1) # Adding Singleton direction
        output = model(xb)   # Forward Propagation..
        output = torch.sigmoid(output.squeeze())

        loss = criteria(output, yb) # Comput Loss
        loss.backward()     # Backward Propagation
        optimizer.step()        # Update Weight

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 1/20 and loss = 0.2737128436565399
epoch = 2/20 and loss = 0.02861355058848858
epoch = 3/20 and loss = 0.1355224847793579
epoch = 4/20 and loss = 0.07109413295984268
epoch = 5/20 and loss = 0.027052080258727074
epoch = 6/20 and loss = 0.001649001962505281
epoch = 7/20 and loss = 0.016037514433264732
epoch = 8/20 and loss = 0.030673502013087273
epoch = 9/20 and loss = 0.014872048981487751
epoch = 10/20 and loss = 0.0005495353252626956
epoch = 11/20 and loss = 0.0051307473331689835
epoch = 12/20 and loss = 0.002585466718301177
epoch = 13/20 and loss = 0.001827781554311514
epoch = 14/20 and loss = 0.00023348921968135983
epoch = 15/20 and loss = 0.0010508488630875945
epoch = 16/20 and loss = 0.0045373872853815556
epoch = 17/20 and loss = 0.005816848948597908
epoch = 18/20 and loss = 0.001892432221211493
epoch = 19/20 and loss = 0.00022673711646348238
epoch = 20/20 and loss = 0.0006848170887678862


In [22]:
# Validation

model.eval()

val_loss = 0
correct = 0
total = 0

with torch.no_grad():

    for xb, yb in test_loader:
        xb = xb.unsqueeze(1)
        output = model(xb)  # Forward Propagation
        output = torch.sigmoid(output.squeeze())
        loss = criteria(output, yb)
        val_loss += loss.item()
        predicted = (output > 0.5).float()
        total += yb.size(0)
        correct += (predicted == yb).sum().item() 
val_loss = val_loss / len(test_loader)
accuracy = correct / total

print(f"Validation Loss:{val_loss}")
print(f"Validation Accuracy: {accuracy*100}")

Validation Loss:0.03636992132265328
Validation Accuracy: 98.85864793678665


In [23]:
# Evaluate 

model.eval()

with torch.no_grad():
    correct_vals = 0
    total = 0

    for xb, yb in test_loader:

        xb = xb.unsqueeze(1)

        output = model(xb)      # Forward Propagation
        predicted = (torch.sigmoid(output.squeeze())> 0.5).float()

        total += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"Accuarcy = {correct_vals/total*100}")

Accuarcy = 98.85864793678665
